https://scikit-learn.org/stable/machine_learning_map.html

### 사과 품질 분류하기
* 컬럼
    - Size : 크기
    - Weight : 무게
    - Sweetness : 단맛
    - Crunchiness : 아삭한 정도
    - Juiciness : 사과 즙의 정도
    - Ripeness : 사과의 숙성 정도
    - Acidity : 신맛
    - Quality : 품질

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data_set/4.분류/apple_quality.csv')
df.head()

,A_id,Size,Weight,Sweetness,Crunchiness,Juiciness,Ripeness,Acidity,Quality
0,0.0,-3.970049,-2.512336,5.346330,-1.012009,1.844900,0.329840,-0.491590483,good
1,1.0,-1.195217,-2.839257,3.664059,1.588232,0.853286,0.867530,-0.722809367,good
2,2.0,-0.292024,-1.351282,-1.738429,-0.342616,2.838636,-0.038033,2.621636473,bad
3,3.0,-0.657196,-2.271627,1.324874,-0.097875,3.637970,-3.413761,0.790723217,good
4,4.0,1.364217,-1.296612,-0.384658,-0.553006,3.030874,-1.303849,0.501984036,good


In [3]:
df['Quality'].unique()

<StringArray>
['good', 'bad', nan]
Length: 3, dtype: str

In [4]:
df.isnull().sum()

A_id           1
Size           1
Weight         1
Sweetness      1
Crunchiness    1
Juiciness      1
Ripeness       1
Acidity        0
Quality        1
dtype: int64

In [5]:
df.shape

(4001, 9)

In [6]:
df.dropna(axis=0, inplace=True)

In [7]:
df.shape

(4000, 9)

In [8]:
df.columns

Index(['A_id', 'Size', 'Weight', 'Sweetness', 'Crunchiness', 'Juiciness',
       'Ripeness', 'Acidity', 'Quality'],
      dtype='str')

In [9]:
df.head(1)

,A_id,Size,Weight,Sweetness,Crunchiness,Juiciness,Ripeness,Acidity,Quality
0,0.0,-3.970049,-2.512336,5.34633,-1.012009,1.8449,0.32984,-0.491590483,good


In [11]:
features = ['Size', 'Weight', 'Sweetness', 'Crunchiness', 'Juiciness',
       'Ripeness', 'Acidity']
label = 'Quality'

X, y = df[features], df[label]
X.head(3)

,Size,Weight,Sweetness,Crunchiness,Juiciness,Ripeness,Acidity
0,-3.970049,-2.512336,5.346330,-1.012009,1.844900,0.329840,-0.491590483
1,-1.195217,-2.839257,3.664059,1.588232,0.853286,0.867530,-0.722809367
2,-0.292024,-1.351282,-1.738429,-0.342616,2.838636,-0.038033,2.621636473


In [12]:
y.head(3)

0    good
1    good
2     bad
Name: Quality, dtype: str

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [15]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2800, 7), (2800,), (1200, 7), (1200,))

### 모델 생성 및 학습, 평가

#### Kneighbors(K-최근접 이웃 알고리즘)
* 주위의 가장 가까운 다른 데이터를 보고 현재 데이터를 판단
* 기본 비교 개수는 5개로 되어 있다
* 비교 개수를 변경하고자 할 경우 n_neighbors에 값 지정

In [16]:
from sklearn.neighbors import KNeighborsClassifier

In [18]:
kn = KNeighborsClassifier() # 모델 생성
kn.fit(X_train, y_train) # 학습
kn.score(X_test, y_test)

0.8883333333333333

#### SVM(Support Vector Machine)
* 특정 데이터들을 구분하는 선을 찾고, 이를 기반으로 패턴을 인식하는 방법
* kernel : linear, rbf
* linear : 선형으로 데이터들을 구분지을 수 있는 경우. 즉, 데이터 분포가 균일 적인 경우
* rbf : 선형으로 데이터를 구분지을 수 없는 경우. 즉, 데이터 분포가 복잡한 경우

In [20]:
import sklearn.svm as svm

In [21]:
svm_linear = svm.SVC(kernel='linear')
svm_linear.fit(X_train, y_train)
svm_linear.score(X_train, y_train)

0.7564285714285715

In [22]:
svm_rbf = svm.SVC(kernel='rbf')
svm_rbf.fit(X_train, y_train)
svm_rbf.score(X_train, y_train)

0.9032142857142857

### DecisionTree
* 특정 조건에 따라 가지치기 과정을 반복하면서 모델을 생성한다

In [23]:
from sklearn.tree import DecisionTreeClassifier

In [26]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt.score(X_train, y_train)

1.0

### Ensemble(앙상블)
* 여러 개의 분류기(알고리즘)를 생성하고 그 예측을 결합함으로써 보다 정확한 분류기 생성
* 앙상블의 대표 알고리즘은 랜덤포레스트와 그래디언트부스팅이 있다
>
* 앙상블 학습의 유형
    - 보팅(Voting) : 서로 다른 알고리즘을 가진 분류기를 결합하여 사용
    - 배깅(Bagging) : 모두 같은 유형의 알고리즘을 가진 분류기를 결합하여 사용
        * 대표적 알고리즘인 랜덤포레스트가 있다
    - 부스팅(Boosting) : 오류를 개선해 나가면서 학습하는 방식(다른 알고리즘에 비해 시간이 더 걸림)

#### Voting(보팅)
* 하드보팅(Hard Voting) : 다수의 결정에 의해 결과값이 선정된다
* 소프트보팅(Soft Voting) : 결정된 값들의 평균을 구하고 가장 높은 값을 선정
* 일반적으로 하드보팅보다 소프트 보팅이 성능이 좋아 소프트 보팅을 많이 사용한다

In [30]:
svm_rbf = svm.SVC( kernel='rbf', probability=True ) # probability : soft voting 시 필요
svm_rbf.fit(X_train, y_train)

kn = KNeighborsClassifier()
kn.fit(X_train, y_train)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [31]:
from sklearn.ensemble import VotingClassifier

In [32]:
vo = VotingClassifier(estimators=[
    ('svc', svm_rbf), ('knn',kn), ('DT', dt)
], voting='soft')

In [33]:
vo.fit(X_train, y_train)
print("vo : ", vo.score(X_test, y_test))
print("svm_rbf : ", svm_rbf.score(X_test, y_test))
print("kn : ", kn.score(X_test, y_test))
print("dt : ", dt.score(X_test, y_test))

vo :  0.865
svm_rbf :  0.8783333333333333
kn :  0.8883333333333333
dt :  0.7958333333333333


#### RandomForest(랜덤포레스트)
* decision tree 알고리즘을 여러 개의 분류기로 만들어서 보팅(소프트보팅)으로 최종 결정한다

In [34]:
from sklearn.ensemble import RandomForestClassifier

In [36]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
rfc.score(X_test, y_test)

0.8716666666666667

#### 부스팅(Boosting)
* GBM(Gradient Boosting Machine)
    - decision tree를 묶어 강력한 model을 만드는 ensemble기법입니다.
    - 순차적으로 학습-예측하면서 잘못 예측한 데이터의 오류를 개선해 나가면서 학습하는 방법
    - 다른 알고리즘에 비해 처리속도가 느림

In [37]:
from sklearn.ensemble import GradientBoostingClassifier

In [38]:
gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)
gbc.score(X_test, y_test)

0.8433333333333334

### 예측

In [39]:
X_test.head(3)

,Size,Weight,Sweetness,Crunchiness,Juiciness,Ripeness,Acidity
196,-0.815840,-0.301607,-0.001498,-0.384304,-1.081999,2.344492,-0.493978613
2868,-0.981949,1.055757,1.823252,-1.659132,-1.166565,-1.587970,1.954753513
920,0.155756,-0.698560,0.766755,1.865081,-0.285543,0.944386,-1.012158006


In [40]:
kn.predict(X_test)

array(['good', 'good', 'good', ..., 'bad', 'bad', 'bad'],
      shape=(1200,), dtype=object)

In [41]:
y_test

196     good
2868    good
920     good
2228    good
2874     bad
        ... 
3861     bad
83      good
686      bad
142      bad
1308     bad
Name: Quality, Length: 1200, dtype: str

In [42]:
kn.predict( [[-0.981949, 1.055757, 1.823252, -1.659132, -1.166565, -1.587970, 1.954753513]] )

C:\Users\user\anaconda3\envs\ai\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


array(['good'], dtype=object)